In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")
payer_table = dbutils.widgets.get("payer_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW insurance_src AS
SELECT
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS STRING) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(InsRank AS INT) AS InsRank,
  CAST(ActiveIns AS INT) AS ActiveIns,
  CAST(InsCode AS STRING) AS InsCode,
  NULL AS InsEDIID,
  NULL AS InsEID,
  CAST(InsGroupID AS STRING) AS InsGroupID,
  CAST(InsPolicy AS STRING) AS InsPolicy,
  NULL AS InsPreCertNbr,
  CAST(InsBalance AS DOUBLE) AS InsBalance,
  CAST(InsPymt AS DOUBLE) AS InsPymt,
  CAST(InsAdj AS DOUBLE) AS InsAdj,
  CAST(InsName AS STRING) AS InsName,
  CAST(InsAddr1 AS STRING) AS InsAddr1,
  CAST(InsAddr2 AS STRING) AS InsAddr2,
  CAST(InsCity AS STRING) AS InsCity,
  CAST(InsState AS STRING) AS InsState,
  CAST(InsZip AS STRING) AS InsZip,
  NULL AS InsCountry,
  CAST(InsProvince AS STRING) AS InsProvince,
  NULL AS InsPhone,
  NULL AS InsEmail,
  CAST(InsSubFName AS STRING) AS InsSubFName,
  NULL AS InsSubMName,
  CAST(InsSubLName AS STRING) AS InsSubLName,
  NULL AS InsSubSuffix,
  CAST(InsSubAddr1 AS STRING) AS InsSubAddr1,
  CAST(InsSubAddr2 AS STRING) AS InsSubAddr2,
  CAST(InsSubCity AS STRING) AS InsSubCity,
  CAST(InsSubState AS STRING) AS InsSubState,
  CAST(InsSubZip AS STRING) AS InsSubZip,
  NULL AS InsSubCountry,
  NULL AS InsSubProvince,
  CAST(InsSubPhone AS STRING) AS InsSubPhone,
  CAST(InsSubDOB AS STRING) AS InsSubDOB,
  CAST(InsSubSSN AS STRING) AS InsSubSSN,
  CAST(InsSubGender AS STRING) AS InsSubGender,
  NULL AS InsSubRelation,
  CAST(InitialBillDate AS STRING) AS InitialBillDate,
  NULL AS LastBillDate,
  NULL AS LastBillSubmitDate,
  NULL AS LastBillType,
  NULL AS LastMediaType,
  NULL AS InsStatusCode,
  NULL AS InsStatusDate,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
  WITH 
  obd AS (
    SELECT DISTINCT
        client_key,
        office_key,
        invoice_number,
        invoice_date_key        AS InitialBillDate,
        account_balance         AS AcctBalance,
        payor_key,
        date_entered_key,
        total_payments,
        total_adjustments
    FROM {source_table}
    WHERE account_balance <> 0
    AND date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
  ),
  client_enriched AS (
    SELECT
        o.office_key,
        CASE
            WHEN UPPER(o.invoice_number) = 'ADV'
                THEN concat('ADV', ' - ', clt.SourceSystemId)
            ELSE o.invoice_number
        END AS InvNum,
        1 AS InsRank,
        CASE
            WHEN clt.PrimaryInsuranceMemberId IS NULL
                  AND clt.SecondaryInsuranceMemberId IS NOT NULL
                THEN clt.SecondaryInsuranceMemberId
            ELSE clt.PrimaryInsuranceMemberId
        END AS InsPolicy,
        o.AcctBalance,
        o.payor_key,
        o.InitialBillDate,
        o.total_payments,
        o.total_adjustments,
        clt.ConformedFirstName,
        clt.ConformedLastName,
        clt.ConformedAddress1,
        clt.ConformedAddress2,
        clt.City,
        clt.ConformedState,
        clt.ConformedZipcode,
        clt.ConformedBirthDate,
        clt.ConformedGender,
        clt.InsuranceGroupId,
        clt.ConformedPhone1,
        clt.ConformedSocialSecurityNumber,
        o.date_entered_key
    FROM obd o
    LEFT JOIN {client_table} clt
        ON o.client_key = clt.ClientKey
  ),
  insurance_cte (
    SELECT
        to_date(CAST(b.date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
        ofc.OfficeNumber                      AS FacilityCode,
          b.InvNum                              AS AcctNbr,
          b.InsRank                             AS InsRank,
          1                                     AS ActiveIns,
          pd.PayerID                            AS InsCode,
          b.InsuranceGroupId                    AS InsGroupID,
          b.InsPolicy                           AS InsPolicy,
          b.AcctBalance                         AS InsBalance,
          b.total_payments                    AS InsPymt,
          b.total_adjustments                 AS InsAdj,
          CASE
              WHEN pd.Name LIKE '%"%'
                  THEN trim(replace(pd.Name, '"', ' '))
              ELSE pd.Name
          END                                   AS InsName,
          pd.Address1                           AS InsAddr1,
          pd.Address2                           AS InsAddr2,
          pd.City                               AS InsCity,
          pd.State                              AS InsState,
          pd.Zip                                AS InsZip,
          CASE
              WHEN pd.PayorProgram LIKE '%"%'
                  THEN trim(replace(pd.PayorProgram, '"', ' '))
              ELSE pd.PayorProgram
          END                                   AS InsProvince,
          trim(replace(b.ConformedFirstName, '"', ' '))
                                                AS InsSubFName,
          trim(replace(b.ConformedLastName, '"', ' '))
                                                AS InsSubLName,
          replace(b.ConformedAddress1, '|', '')
                                                AS InsSubAddr1,
          trim(replace(b.ConformedAddress2, '"', ' '))
                                                AS InsSubAddr2,
          b.City                                AS InsSubCity,
          b.ConformedState                      AS InsSubState,
          b.ConformedZipcode                    AS InsSubZip,
          b.ConformedPhone1                     AS InsSubPhone,
          replace(CAST(b.ConformedBirthDate AS DATE), '-', '')
                                                AS InsSubDOB,
          b.ConformedSocialSecurityNumber       AS InsSubSSN,
          CASE
              WHEN b.ConformedGender IN ('Male','Female')
                  THEN left(b.ConformedGender, 1)
              ELSE NULL
          END                                   AS InsSubGender,
          concat(
              left(b.InitialBillDate, 4), '-',
              substr(b.InitialBillDate, 5, 2), '-',
              right(b.InitialBillDate, 2)
          )                                     AS InitialBillDate,
          0                                     AS SourceSystemKey
    FROM client_enriched b
    LEFT JOIN {office_table} ofc
        ON ofc.OfficeKey = b.office_key
    LEFT JOIN {payer_table} pd
        ON pd.PayerKey = b.payor_key
  ),
  insurance_clean(
    SELECT *,
    ROW_NUMBER() OVER (
          PARTITION BY AcctNbr
          ORDER BY AcctNbr, InsSubDOB
      ) AS rn
    FROM insurance_cte
  )
  SELECT 
    ReportingDate, FacilityCode, AcctNbr, InsRank, ActiveIns,
    InsCode, InsGroupID, InsPolicy, InsBalance, InsPymt, InsAdj,
    InsName, InsAddr1, InsAddr2, InsCity, InsState, InsZip, InsProvince,
    InsSubFName, InsSubLName, InsSubAddr1, InsSubAddr2, InsSubCity,
    InsSubState, InsSubZip, InsSubPhone, InsSubDOB, InsSubSSN,
    InsSubGender, InitialBillDate, SourceSystemKey
  FROM insurance_clean
  WHERE rn=1
) AS src
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING insurance_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.InsRank = src.InsRank,
    tgt.ActiveIns = src.ActiveIns,
    tgt.InsCode = src.InsCode,
    tgt.InsEDIID = src.InsEDIID,
    tgt.InsEID = src.InsEID,
    tgt.InsGroupID = src.InsGroupID,
    tgt.InsPolicy = src.InsPolicy,
    tgt.InsPreCertNbr = src.InsPreCertNbr,
    tgt.InsBalance = src.InsBalance,
    tgt.InsPymt = src.InsPymt,
    tgt.InsAdj = src.InsAdj,
    tgt.InsName = src.InsName,
    tgt.InsAddr1 = src.InsAddr1,
    tgt.InsAddr2 = src.InsAddr2,
    tgt.InsCity = src.InsCity,
    tgt.InsState = src.InsState,
    tgt.InsZip = src.InsZip,
    tgt.InsCountry = src.InsCountry,
    tgt.InsProvince = src.InsProvince,
    tgt.InsPhone = src.InsPhone,
    tgt.InsEmail = src.InsEmail,
    tgt.InsSubFName = src.InsSubFName,
    tgt.InsSubMName = src.InsSubMName,
    tgt.InsSubLName = src.InsSubLName,
    tgt.InsSubSuffix = src.InsSubSuffix,
    tgt.InsSubAddr1 = src.InsSubAddr1,
    tgt.InsSubAddr2 = src.InsSubAddr2,
    tgt.InsSubCity = src.InsSubCity,
    tgt.InsSubState = src.InsSubState,
    tgt.InsSubZip = src.InsSubZip,
    tgt.InsSubCountry = src.InsSubCountry,
    tgt.InsSubProvince = src.InsSubProvince,
    tgt.InsSubPhone = src.InsSubPhone,
    tgt.InsSubDOB = src.InsSubDOB,
    tgt.InsSubSSN = src.InsSubSSN,
    tgt.InsSubGender = src.InsSubGender,
    tgt.InsSubRelation = src.InsSubRelation,
    tgt.InitialBillDate = src.InitialBillDate,
    tgt.LastBillDate = src.LastBillDate,
    tgt.LastBillSubmitDate = src.LastBillSubmitDate,
    tgt.LastBillType = src.LastBillType,
    tgt.LastMediaType = src.LastMediaType,
    tgt.InsStatusCode = src.InsStatusCode,
    tgt.InsStatusDate = src.InsStatusDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    InsRank,
    ActiveIns,
    InsCode,
    InsEDIID,
    InsEID,
    InsGroupID,
    InsPolicy,
    InsPreCertNbr,
    InsBalance,
    InsPymt,
    InsAdj,
    InsName,
    InsAddr1,
    InsAddr2,
    InsCity,
    InsState,
    InsZip,
    InsCountry,
    InsProvince,
    InsPhone,
    InsEmail,
    InsSubFName,
    InsSubMName,
    InsSubLName,
    InsSubSuffix,
    InsSubAddr1,
    InsSubAddr2,
    InsSubCity,
    InsSubState,
    InsSubZip,
    InsSubCountry,
    InsSubProvince,
    InsSubPhone,
    InsSubDOB,
    InsSubSSN,
    InsSubGender,
    InsSubRelation,
    InitialBillDate,
    LastBillDate,
    LastBillSubmitDate,
    LastBillType,
    LastMediaType,
    InsStatusCode,
    InsStatusDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.InsRank,
    src.ActiveIns,
    src.InsCode,
    src.InsEDIID,
    src.InsEID,
    src.InsGroupID,
    src.InsPolicy,
    src.InsPreCertNbr,
    src.InsBalance,
    src.InsPymt,
    src.InsAdj,
    src.InsName,
    src.InsAddr1,
    src.InsAddr2,
    src.InsCity,
    src.InsState,
    src.InsZip,
    src.InsCountry,
    src.InsProvince,
    src.InsPhone,
    src.InsEmail,
    src.InsSubFName,
    src.InsSubMName,
    src.InsSubLName,
    src.InsSubSuffix,
    src.InsSubAddr1,
    src.InsSubAddr2,
    src.InsSubCity,
    src.InsSubState,
    src.InsSubZip,
    src.InsSubCountry,
    src.InsSubProvince,
    src.InsSubPhone,
    src.InsSubDOB,
    src.InsSubSSN,
    src.InsSubGender,
    src.InsSubRelation,
    src.InitialBillDate,
    src.LastBillDate,
    src.LastBillSubmitDate,
    src.LastBillType,
    src.LastMediaType,
    src.InsStatusCode,
    src.InsStatusDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)